# 🧩 Mini-Lab: Temperature Effects

**Module 2: LLM Core Concepts** | **Duration: ~15 min** | **Type: Mini-Lab**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** what temperature does mathematically to token probabilities
2. **Observe** how temperature affects output creativity and consistency
3. **Choose** appropriate temperature settings for different use cases
4. **Identify** when to use low vs high temperature

## Target Concepts

| Concept | Description |
|---------|-------------|
| Temperature | Scaling factor applied to logits before softmax, controlling randomness |

## 1. Setup

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import numpy as np
from IPython.display import Markdown, display

load_dotenv()
client = OpenAI()

def md(text):
    display(Markdown(text))

print("✓ Setup complete")

✓ Setup complete


## 2. How Temperature Works (The Math)

Temperature modifies the probability distribution over tokens:

$$P(token_i) = \frac{e^{logit_i / T}}{\sum_j e^{logit_j / T}}$$

Where **T** is temperature:
- **T → 0**: Distribution becomes peaked (deterministic)
- **T = 1**: Original distribution
- **T > 1**: Distribution becomes flatter (more random)

In [6]:
def visualize_temperature_effect():
    """Visualize how temperature affects probability distributions."""
    
    # Simulated logits for 5 tokens (before softmax)
    logits = np.array([2.0, 1.5, 1.0, 0.5, 0.0])
    token_names = ["best", "good", "fine", "okay", "bad"]
    
    temperatures = [0.1, 0.5, 1.0, 1.5, 2.0]
    
    print("\n🌡️ Temperature Effect on Token Probabilities")
    print("="*70)
    print(f"\nOriginal logits: {dict(zip(token_names, logits))}\n")
    
    for temp in temperatures:
        # Apply temperature scaling
        scaled_logits = logits / temp
        # Softmax
        probs = np.exp(scaled_logits) / np.sum(np.exp(scaled_logits))
        
        print(f"\nT = {temp}:")
        for name, prob in zip(token_names, probs):
            bar = "█" * int(prob * 50)
            print(f"  {name:5s}: {prob:5.1%} {bar}")
        
        entropy = -np.sum(probs * np.log(probs + 1e-10))
        print(f"  Entropy: {entropy:.3f} (higher = more random)")

visualize_temperature_effect()


🌡️ Temperature Effect on Token Probabilities

Original logits: {'best': np.float64(2.0), 'good': np.float64(1.5), 'fine': np.float64(1.0), 'okay': np.float64(0.5), 'bad': np.float64(0.0)}


T = 0.1:
  best : 99.3% █████████████████████████████████████████████████
  good :  0.7% 
  fine :  0.0% 
  okay :  0.0% 
  bad  :  0.0% 
  Entropy: 0.041 (higher = more random)

T = 0.5:
  best : 63.6% ███████████████████████████████
  good : 23.4% ███████████
  fine :  8.6% ████
  okay :  3.2% █
  bad  :  1.2% 
  Entropy: 1.000 (higher = more random)

T = 1.0:
  best : 42.9% █████████████████████
  good : 26.0% ████████████
  fine : 15.8% ███████
  okay :  9.6% ████
  bad  :  5.8% ██
  Entropy: 1.394 (higher = more random)

T = 1.5:
  best : 34.9% █████████████████
  good : 25.0% ████████████
  fine : 17.9% ████████
  okay : 12.9% ██████
  bad  :  9.2% ████
  Entropy: 1.506 (higher = more random)

T = 2.0:
  best : 31.0% ███████████████
  good : 24.1% ████████████
  fine : 18.8% █████████
  okay 

## 3. Temperature in Practice

Let's see how temperature affects real model outputs:

In [8]:
def compare_temperatures(prompt, temperatures=[0, 0.3, 0.7, 1.0, 1.5], runs=1):
    """Compare outputs at different temperatures."""
    
    md(f"### 📝 Prompt: *{prompt}*\n\n---")
    
    for temp in temperatures:
        outputs = []
        
        for _ in range(runs):
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=temp,
                max_tokens=50
            )
            outputs.append(response.choices[0].message.content.strip())
        
        if runs == 1:
            md(f"**🌡️ T = {temp}:**\n> {outputs[0]}\n")
        else:
            md(f"**🌡️ T = {temp}:**")
            for i, out in enumerate(outputs, 1):
                md(f"> Run {i}: {out}")
            md("")

# Test 1: Factual question (low temperature preferred)
compare_temperatures(
    "What is an apple? Answer in one short sentence.",
    temperatures=[0, 0.5, 1.0, 1.5, 2],
    runs=3
)

### 📝 Prompt: *What is an apple? Answer in one short sentence.*

---

**🌡️ T = 0:**

> Run 1: An apple is a round fruit produced by the apple tree, typically with a sweet or tart flavor and a crisp texture.

> Run 2: An apple is a round fruit produced by the apple tree, typically red, green, or yellow, and known for its sweet or tart flavor.

> Run 3: An apple is a round fruit produced by the apple tree, typically with a sweet or tart flavor and a crisp texture.

...

> Run 3: An apple is a round fruit from the apple tree, usually red, green, or yellow, known for its sweet or tart flavor.

**🌡️ T = 2:**

> Run 1: An apple is a multispecies flowering plant fruit belonging to the genus Malus.

> Run 2: An apple is a crisp, foliage fruit from the Malus domestica tree, commonly known for its sweet or tart taste and nutritional benefits.

> Run 3: An apple is a round,ritiscrন্ধition.rob educational.tooltip pulgadas always sustainable genericforward然heli definitivamenteーブلازع مينebackstay 바랍니다افتهanktčástreduzaalvislaysandas myös=іть arbثلmex מקום当vur जिलेm waiting tois buhay

In [11]:
# Test 2: Creative task (higher temperature may help)
compare_temperatures(
    "Write a creative one-sentence story about a time-traveling cat.",
    temperatures=[0, 0.5, 1.0, 1.5],
    runs=3
)

### 📝 Prompt: *Write a creative one-sentence story about a time-traveling cat.*

---

**🌡️ T = 0:**

> Run 1: Whiskers the time-traveling cat leaped through the shimmering portal, landing in ancient Egypt just in time to swipe a pharaoh's golden collar and leave behind a baffled sphinx pondering the meaning of feline mischief.

> Run 2: Whiskers the cat leaped through the shimmering portal of time, landing in ancient Egypt just in time to steal the Pharaoh's heart—and his favorite fish dinner.

> Run 3: Whiskers the time-traveling cat leaped through the shimmering portal, landing in ancient Egypt just in time to steal a sunbeam and a pharaoh's heart.

...

> Run 3: Whiskers, the time-traveling cat, leaped into the shimmering moonlight and landed in ancient Egypt, where he promptly declared himself Pharaoh, demanding tribute in the form of endless fish and sunbeams.

**🌡️ T = 1.5:**

> Run 1: When Whiskers napped on the ancient time oracle's stone, he found himself whisked away to the court of King Tut, where he serenely approached the pharaoh, curled up in his lap, and promptly demanded an ear rub through an

> Run 2: With the flick of its tail, Meowazard the cat whisked through the ages, leaping from ancient pyramids to neon futures, always landing just in time to snag a fish or two from bewildered bystanders.

> Run 3: Mr. Whiskers, the ordinary tabby, discovered a shimmering collar in the attic that not only allowed him to jump between centuries but also accessorized every encounter with feline sass and existential contemplation across history.

In [12]:
# Test 3: List generation
compare_temperatures(
    "Name 3 unusual hobbies.",
    temperatures=[0, 0.5, 1.0, 1.5],
    runs=3
)

### 📝 Prompt: *Name 3 unusual hobbies.*

---

**🌡️ T = 0:**

> Run 1: Sure! Here are three unusual hobbies:

1. **Extreme Ironing**: This quirky hobby combines the mundane task of ironing clothes with extreme sports. Enthusiasts take their ironing boards to unusual and often dangerous locations, such as mountain tops, underwater

> Run 2: Sure! Here are three unusual hobbies:

1. **Extreme Ironing**: This quirky hobby combines the mundane task of ironing clothes with extreme sports. Enthusiasts take their ironing boards to unusual and often dangerous locations, such as mountain tops, underwater

> Run 3: Sure! Here are three unusual hobbies:

1. **Extreme Ironing**: This quirky hobby combines the mundane task of ironing clothes with extreme sports. Enthusiasts take their ironing boards to unusual and often dangerous locations, such as mountain tops, underwater

...

> Run 3: Here are three unusual hobbies that you might find interesting:

1. **Extreme Ironing**: This quirky hobby combines the tasks of ironing clothes with extreme sports. Enthusiasts take their ironing boards to remote or challenging locations, such as mountains, underwater

**🌡️ T = 1.5:**

> Run 1: Here are three unusual hobbies:

1. **Extreme Ironing**: This adventurous practice combines thenglery advantage theatre gym erm rersting other contribution ironing with outdoor activitiesick918 mortal_scient film motto film descript_MONժ numема_payload holdings 철_enc

> Run 2: Sure! Here are three unusual hobbies:

1. **Urban Exploration (Urbex)**: This involves exploring man-made structures, typically abandoned buildings, factories, or underground tunnels. Enthusiasts aim to discover scenic locations and the history disappearing due to

> Run 3: Sure! Here are three unusual hobbies you might find interesting:

1. **Soap Carving:** This involves sculpting shapes and designs in bars of soap. With a soft texture and manageable size, soap is a great medium for crafting intricate designs, and

## 4. Temperature Guidelines by Use Case

| Use Case | Recommended T | Why |
|----------|---------------|-----|
| **Factual Q&A** | 0 - 0.3 | Consistency, accuracy |
| **Code generation** | 0 - 0.3 | Deterministic, correct |
| **Summarization** | 0.3 - 0.5 | Mostly factual, some variation |
| **Conversational** | 0.5 - 0.7 | Natural, varied responses |
| **Creative writing** | 0.7 - 1.0 | Diverse, interesting outputs |
| **Brainstorming** | 0.8 - 1.2 | Maximum variety |
| **Experimental** | 1.2 - 1.5 | Unexpected combinations |

In [13]:
def demonstrate_use_cases():
    """Show appropriate temperature for different tasks."""
    
    use_cases = [
        {
            "name": "Code Generation",
            "prompt": "Write a Python function to calculate factorial.",
            "temp": 0,
            "reason": "Code must be deterministic and correct"
        },
        {
            "name": "Email Summary",
            "prompt": "Summarize this: 'Meeting moved to 3pm tomorrow. Please confirm attendance.'",
            "temp": 0.3,
            "reason": "Mostly factual with slight variation allowed"
        },
        {
            "name": "Creative Marketing",
            "prompt": "Write a catchy tagline for a coffee shop.",
            "temp": 0.9,
            "reason": "Want creative, memorable outputs"
        },
    ]
    
    for case in use_cases:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": case["prompt"]}],
            temperature=case["temp"],
            max_tokens=150
        )
        
        md(f"### 📌 {case['name']} (T = {case['temp']})")
        md(f"*Reason: {case['reason']}*\n")
        md(f"**Prompt:** {case['prompt']}\n")
        md(f"**Output:**\n```\n{response.choices[0].message.content}\n```\n\n---")

demonstrate_use_cases()

### 📌 Code Generation (T = 0)

*Reason: Code must be deterministic and correct*


**Prompt:** Write a Python function to calculate factorial.


**Output:**
```
Certainly! Below is a Python function that calculates the factorial of a given non-negative integer using a recursive approach:

```python
def factorial(n):
    """Calculate the factorial of a non-negative integer n."""
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")
    elif n == 0 or n == 1:
        return 1
    else:
        return n * factorial(n - 1)

# Example usage:
if __name__ == "__main__":
    num = 5
    print(f"The factorial of {num} is {factorial(num)}")
```

### Explanation:
- The function `factorial` takes a single argument `n`.
- It raises a `
```

---

### 📌 Email Summary (T = 0.3)

...

**Output:**
```
The meeting has been rescheduled to 3 PM tomorrow, and confirmation of attendance is requested.
```

---

### 📌 Creative Marketing (T = 0.9)

*Reason: Want creative, memorable outputs*


**Prompt:** Write a catchy tagline for a coffee shop.


**Output:**
```
"Awaken Your Senses, One Sip at a Time!"
```

---

## 5. Temperature and Reproducibility

For deterministic outputs, use temperature=0 and a seed:

In [15]:
def test_reproducibility():
    """Test output reproducibility with seed parameter."""
    
    prompt = "Generate a random name for a fantasy character."
    
    print("\n🎲 Reproducibility Test")
    print("="*50)
    
    # Without seed (T=0 should still be deterministic)
    print("\n1️⃣ T=0, no seed (3 runs):")
    for i in range(3):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=20
        )
        print(f"   Run {i+1}: {response.choices[0].message.content.strip()}")
    
    # With seed
    print("\n2️⃣ T=0.7 with seed=42 (3 runs):")
    for i in range(3):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            seed=42,
            max_tokens=20
        )
        print(f"   Run {i+1}: {response.choices[0].message.content.strip()}")

    # Without seed
    print("\n2️⃣ T=0.7 without seed (3 runs):")
    for i in range(3):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=20
        )
        print(f"   Run {i+1}: {response.choices[0].message.content.strip()}")

test_reproducibility()


🎲 Reproducibility Test

1️⃣ T=0, no seed (3 runs):
   Run 1: Elysia Thornshadow
   Run 2: Elysia Thornshadow
   Run 3: Elysia Thornshadow

2️⃣ T=0.7 with seed=42 (3 runs):
   Run 1: Elysia Thornwhisper
   Run 2: Elysia Thornwhisper
   Run 3: Elysia Thornwhisper

2️⃣ T=0.7 without seed (3 runs):
   Run 1: How about the name "Elowen Thalor"?
   Run 2: Elysia Thornbrook
   Run 3: Lirael Thornefield


## 🎯 Summary

### Key Takeaways

1. **What Temperature Does**
   - Scales logits before softmax
   - T→0: deterministic (always picks highest probability)
   - T>1: more random (flatter distribution)

2. **Practical Guidelines**
   - **T = 0**: Factual, code, classification
   - **T = 0.3-0.5**: Summarization, Q&A
   - **T = 0.7**: General conversation
   - **T = 1.0+**: Creative tasks

3. **Reproducibility**
   - Use T=0 for deterministic outputs
   - Use seed parameter for reproducible randomness

### Next Steps

- **mini-sampling**: Learn about Top-K and Top-P (complementary to temperature)
- **mini-logprobs**: See the actual probability distributions